In [ ]:
!pip install -q sarvamai gradio

In [ ]:
   import os, time, io, zipfile, urllib.request

In [ ]:
   def extract_text(path) -> str:
       name = os.path.basename(path)
       ext = name.lower().rsplit(".", 1)[-1]
       mime = {"pdf": "application/pdf", "png": "image/png",
               "jpg": "image/jpeg", "jpeg": "image/jpeg"}.get(ext, "application/octet-stream")
       with open(path, "rb") as f:
           job = client.doc_ai.digitise(
               file=[(name, f, mime)],
               language="en-IN",
               output_format="md",
           )
       TERMINAL = {"completed", "partially_completed", "failed", "rejected"}
       while True:
           st = client.doc_ai.get_status(job_id=job.job_id)
           if st.status.lower() in TERMINAL:
               break
           time.sleep(3)
       if st.status.lower() not in ("completed", "partially_completed"):
           raise RuntimeError(f"Document job ended with status: {st.status}")
       dl = client.doc_ai.get_download_url(job_id=job.job_id)
       data = urllib.request.urlopen(dl.url).read()
       z = zipfile.ZipFile(io.BytesIO(data))
       md_names = sorted(n for n in z.namelist() if n.lower().endswith(".md"))
       return "\n".join(z.read(n).decode("utf-8") for n in md_names)

In [ ]:
def ask_llm(prompt):
    r = client.chat.completions(model="sarvam-105b-conversations",
                                messages=[{"role": "user", "content": prompt}])
    raw = (r.choices[0].message.content or "").strip()
    raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    return re.sub(r"\s*```$", "", raw)

def analyse(text, language):
    raw = ""
    try:
        raw = ask_llm(CLAUSE_PROMPT.replace("LANGUAGE", language) + text)
        start, end = raw.find("{"), raw.rfind("}")
        return json.loads(raw[start:end + 1])["clauses"], ""
    except Exception as e:
        print("ANALYSE ERROR:", repr(e))
        print("RAW OUTPUT:", raw[:1500])
        return [], f"{type(e).__name__}: {e}\n\nRaw model output:\n{raw[:800]}"

In [ ]:
import os, re, json, base64, tempfile, time, io, zipfile, urllib.request
import gradio as gr
from sarvamai import SarvamAI
from google.colab import userdata

client = SarvamAI(api_subscription_key=userdata.get("SARVAM_API_KEY"))
LANGS = {"Hindi": "hi-IN", "Telugu": "te-IN", "Tamil": "ta-IN",
         "Marathi": "mr-IN", "Bengali": "bn-IN", "English": "en-IN"}
ICON = {"red": "🔴", "amber": "🟡", "green": "🟢"}

CLAUSE_PROMPT = """You are a contract-risk assistant for tenants in India.
Below is the text of a rental agreement.
Reply ONLY with valid JSON, no markdown fences, in this shape:
{"clauses":[{"type":"lock_in|deposit|penalty|rent_hike|eviction|repairs|other",
"source_line":"exact sentence copied word for word from the contract",
"explanation":"2 short plain sentences a first-time tenant understands",
"risk":"red|amber|green"}]}
Rules: return the 4 most important clauses, riskiest first. Never invent a clause.
Write every explanation in LANGUAGE.

CONTRACT:
"""

def extract_text(path) -> str:
    name = os.path.basename(path)
    ext = name.lower().rsplit(".", 1)[-1]
    mime = {"pdf": "application/pdf", "png": "image/png",
            "jpg": "image/jpeg", "jpeg": "image/jpeg"}.get(ext, "application/octet-stream")
    with open(path, "rb") as f:
        job = client.doc_ai.digitise(
            file=[(name, f, mime)],
            language="en-IN",
            output_format="md",
        )
    TERMINAL = {"completed", "partially_completed", "failed", "rejected"}
    while True:
        st = client.doc_ai.get_status(job_id=job.job_id)
        if st.status.lower() in TERMINAL:
            break
        time.sleep(3)
    if st.status.lower() not in ("completed", "partially_completed"):
        raise RuntimeError(f"Document job ended with status: {st.status}")
    dl = client.doc_ai.get_download_url(job_id=job.job_id)
    data = urllib.request.urlopen(dl.url).read()
    z = zipfile.ZipFile(io.BytesIO(data))
    md_names = sorted(n for n in z.namelist() if n.lower().endswith(".md"))
    return "\n".join(z.read(n).decode("utf-8") for n in md_names)

def ask_llm(prompt):
    r = client.chat.completions(model="sarvam-105b-conversations",
                                messages=[{"role": "user", "content": prompt}])
    raw = (r.choices[0].message.content or "").strip()
    raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    return re.sub(r"\s*```$", "", raw)

def analyse(text, language):
    raw = ""
    try:
        raw = ask_llm(CLAUSE_PROMPT.replace("LANGUAGE", language) + text)
        start, end = raw.find("{"), raw.rfind("}")
        return json.loads(raw[start:end + 1])["clauses"], ""
    except Exception as e:
        print("ANALYSE ERROR:", repr(e))
        print("RAW OUTPUT:", raw[:1500])
        return [], f"{type(e).__name__}: {e}\n\nRaw model output:\n{raw[:800]}"

def speak(text, lang_code):
    r = client.text_to_speech.convert(text=text[:500], model="bulbul:v4-flash",
                                      language_code=lang_code, speaker="anushka")
    f = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
    f.write(base64.b64decode("".join(r.audios)))
    f.close()
    return f.name

EMPTY = gr.update(choices=[], value=None)

def run(file, pasted, language):
    try:
        text = pasted.strip() if pasted and pasted.strip() else (extract_text(file) if file else "")
    except Exception as e:
        return f"Could not read the document.\n\n```\n{type(e).__name__}: {e}\n```", [], EMPTY, ""
    if not text:
        return "Upload a file or paste contract text.", [], EMPTY, ""
    clauses, err = analyse(text, language)
    if not clauses:
        return "Couldn't read the clauses.\n\n```\n" + err + "\n```", [], EMPTY, text
    md = "\n\n".join(
        f"### {ICON.get(c.get('risk'), '⚪')} {str(c.get('type', 'clause')).replace('_', ' ').title()}\n"
        f"{c.get('explanation', '')}\n\n> {c.get('source_line', '')}" for c in clauses)
    labels = [f"{i+1}. {c.get('type', 'clause')}" for i, c in enumerate(clauses)]
    return md, clauses, gr.update(choices=labels, value=labels[0]), text

def listen(label, clauses, language):
    if not label or not clauses:
        return None
    return speak(clauses[int(label.split(".")[0]) - 1]["explanation"], LANGS[language])

def what_if(q, clauses, language):
    if not q or not clauses:
        return "Analyse a contract first."
    return ask_llm("Answer ONLY from these clauses and quote the clause you used. "
                   "If the contract doesn't say, answer 'the contract doesn't say'. "
                   f"Reply in {language}.\n\nCLAUSES: {json.dumps(clauses, ensure_ascii=False)}\n\nQUESTION: {q}")

with gr.Blocks(title="Sign Se Pehle") as demo:
    gr.Markdown("# साइन से पहले — Sign Se Pehle\n*Not legal advice. Check important clauses with a professional.*")
    clauses_state = gr.State([])
    language = gr.Dropdown(list(LANGS), value="Hindi", label="Explain in")
    file = gr.File(label="Photo or PDF of the agreement", type="filepath")
    pasted = gr.Textbox(label="...or paste contract text (to test without Vision)", lines=4)
    go = gr.Button("Analyse", variant="primary")
    out = gr.Markdown()
    extracted = gr.Textbox(label="Text read from the document", lines=5)
    which = gr.Dropdown(label="Pick a clause to hear", choices=[])
    play = gr.Button("🔊 Listen")
    audio = gr.Audio(label="Explanation")
    q = gr.Textbox(label="What if...? Ask about your contract")
    ask = gr.Button("Ask")
    ans = gr.Markdown()

    go.click(run, [file, pasted, language], [out, clauses_state, which, extracted])
    play.click(listen, [which, clauses_state, language], audio)
    ask.click(what_if, [q, clauses_state, language], ans)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7f7edaa687b1eb1554.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ANALYSE ERROR: ApiError()
RAW OUTPUT: 
ANALYSE ERROR: ApiError()
RAW OUTPUT: 
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7f7edaa687b1eb1554.gradio.live
